# ¿Escala fija o escala que evoluciona? Modelo de masa refinado y galaxias lejanas

**Proyecto:** *Vacuum Particle Journey*, línea A (mecanismo), notebook 13.

**De dónde venimos (notebook 12).** La ley térmica de Bose, $a = g_N/(1-e^{-\sqrt{g_N/s}})$, con $s = a_\Lambda/2\pi$ fijada por la energía oscura y sin parámetros libres, pasó los cuatro criterios preregistrados en SPARC. Quedaban dos dudas: (1) todas las leyes compartían una ondulación de residuos que apunta al modelo de masa estelar, y la escala se mueve con él; (2) el grupo reservado no distinguía entre $a_\Lambda/2\pi$ y $cH_0/2\pi$.

**Dos variantes de la misma idea**, que solo se distinguen con el tiempo cósmico:

| Variante | Escala | Origen físico | Hoy | A z = 1 |
|---|---|---|---|---|
| **Λ** | $s = c\sqrt{\Lambda/3}\,/\,2\pi$ | horizonte de de Sitter (energía oscura pura) | 8.6·10⁻¹¹ | igual |
| **H** | $s(z) = cH(z)\,/\,2\pi$ | horizonte aparente del universo en expansión, con $k_BT = \hbar H/2\pi$ (Cai y Kim 2005) | 10.4·10⁻¹¹ | ×1.79 |

**Aviso de transparencia.** En 2026 se publicó una medida de la escala de la RAR a redshift intermedio (MUSE-DARK III, Ciocan et al., A&A 709, L16): crece con z. La conocimos **antes** de escribir este notebook, así que la parte B **no es una prueba a ciegas**: es una confrontación con datos publicados. La única prueba a ciegas que queda es la de la tabla de predicciones de la sección 7, frente a datos futuros.

## Plan
- **Parte A (SPARC):** dar a cada galaxia su propia masa del bulbo, distancia e inclinación (con las incertidumbres del catálogo) y ver si la ondulación desaparece y adónde se mueve la escala.
- **Parte B (galaxias lejanas):** comparar la forma de $s(z)$ medida con las dos variantes.
- **Parte C:** predicciones numéricas para z = 0.5–3.

In [ ]:
import os, io, re, glob, zipfile, urllib.request, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares, minimize_scalar
from scipy.special import i0, i1, k0, k1
from scipy.stats import spearmanr

warnings.filterwarnings("ignore", category=RuntimeWarning)
RNG = np.random.default_rng(42)
QUICK = False

COL = {"Datos": "#222222", "Solo bariones": "#8a8a85", "NFW ΛCDM (c–M)": "#6250d6",
       "RAR (g† = 1.2e-10)": "#e87ba4", "RAR local (g† global)": "#eb6834", "RAR no local (ER)": "#1baf7a", "Instrumental": "#2a78d6", "Densidad": "#eda100", "Tamaño": "#e34948"}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.edgecolor": "#c3c2b7", "lines.linewidth": 2, "font.size": 10, "legend.frameon": False,
})

G = 4.30091e-6
C_KMS = 299792.458
H0 = 70.0; H0_KPC = H0 / 1000.0
KPC_M = 3.0857e19
TO_MS2 = 1e6 / KPC_M                    # (km/s)^2/kpc -> m/s^2
try:
    import plotly.graph_objects as go
    HAVE_PLOTLY = True
except ImportError:
    HAVE_PLOTLY = False
    print("plotly no disponible: las figuras 3D se harán con matplotlib (estáticas)")
from scipy import fft as sfft
N_GRID = 128
H0_PLANCK, OMEGA_M, OMEGA_L = 67.4, 0.315, 0.685
C_MS = 2.99792458e8; MPC_M = 3.0857e22; TO_MS2 = 1e6 / 3.0857e19
cH = lambda H0: C_MS * H0 * 1e3 / MPC_M
E = lambda z: np.sqrt(OMEGA_M * (1 + np.asarray(z, float)) ** 3 + OMEGA_L)
S_LAMBDA = cH(H0_PLANCK) * np.sqrt(OMEGA_L) / (2 * np.pi)
S_H0 = cH(H0_PLANCK) / (2 * np.pi)
print(f"Variante Λ: s = {S_LAMBDA:.3e} m/s² (constante)")
print(f"Variante H: s(z) = {S_H0:.3e} · E(z) m/s²")

## 1. Datos SPARC y metadatos (distancia, inclinación y sus errores)

In [ ]:
DATA_DIR = "sparc_rotmod"
BASES = ["https://astroweb.case.edu/SPARC/", "http://astroweb.cwru.edu/SPARC/"]
ZENODO = "https://zenodo.org/records/16284118/files/Rotmod_LTG.zip?download=1"

def _get(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    return urllib.request.urlopen(req, timeout=60).read()

def _files():
    return sorted(glob.glob(os.path.join(DATA_DIR, "**", "*_rotmod.dat"), recursive=True))

def get_sparc():
    os.makedirs(DATA_DIR, exist_ok=True)
    if _files():
        return True
    for url in [b + "Rotmod_LTG.zip" for b in BASES] + [ZENODO]:
        try:
            print("Descargando", url)
            zipfile.ZipFile(io.BytesIO(_get(url))).extractall(DATA_DIR)
            if _files():
                return True
        except Exception as e:
            print("  falló:", type(e).__name__)
    try:
        from google.colab import files
        print("Sube Rotmod_LTG.zip (de astroweb.case.edu/SPARC)")
        for name, content in files.upload().items():
            zipfile.ZipFile(io.BytesIO(content)).extractall(DATA_DIR)
        return bool(_files())
    except Exception:
        return False

def get_quality():
    for b in BASES:
        try:
            txt = _get(b + "SPARC_Lelli2016c.mrt").decode("latin-1")
            q = {}
            for line in txt.splitlines():
                t = line.split()
                if len(t) >= 18 and re.fullmatch(r"\d+", t[1] or "") and t[17] in ("1", "2", "3"):
                    q[t[0]] = dict(Q=int(t[17]), Inc=float(t[5]))
            if len(q) > 100:
                return q
        except Exception:
            pass
    return None

def load_rotmod(path):
    name = os.path.basename(path).replace("_rotmod.dat", "")
    d = np.atleast_2d(np.loadtxt(path, comments="#"))
    g = dict(name=name, r=d[:, 0], vobs=d[:, 1], err=d[:, 2], vgas=d[:, 3], vdisk=d[:, 4], vbul=d[:, 5])
    ok = (g["r"] > 0) & (g["err"] > 0)
    o = np.argsort(g["r"][ok])
    for k in ["r", "vobs", "err", "vgas", "vdisk", "vbul"]:
        g[k] = g[k][ok][o]
    g["vgas2"] = g["vgas"] * np.abs(g["vgas"])
    g["vstar2"] = g["vdisk"] * np.abs(g["vdisk"]) + 1.4 * g["vbul"] * np.abs(g["vbul"])
    return g

def _v2_expdisk(r, M, Rd):
    y = r / (2 * Rd)
    return 4 * np.pi * G * (M / (2 * np.pi * Rd**2)) * Rd * y**2 * (i0(y) * k0(y) - i1(y) * k1(y))

def _v2_nfw(r, V200, c):
    x = r / (V200 / (10 * H0_KPC))
    mu = lambda y: np.log1p(y) - y / (1 + y)
    return V200**2 * mu(c * x) / (x * mu(c))

def make_synthetic_set(n=30, seed=0):
    rng = np.random.default_rng(seed)
    out = {}
    for name in ["NGC3198", "NGC2403", "DDO154"] + [f"SYN{i:03d}" for i in range(n - 3)]:
        V200 = 10 ** rng.uniform(1.6, 2.4); c = 10 ** rng.normal(1.0, 0.12)
        Rd = rng.uniform(1, 5); Ml = 10 ** rng.uniform(8.5, 11); Mg = Ml * 10 ** rng.uniform(-1, 0.7)
        r = np.linspace(0.3 * Rd, rng.uniform(5, 9) * Rd, rng.integers(12, 40))
        vdisk = np.sqrt(_v2_expdisk(r, Ml, Rd)); vgas = np.sqrt(_v2_expdisk(r, Mg, 2.5 * Rd))
        vtrue = np.sqrt(vgas**2 + 0.5 * vdisk**2 + _v2_nfw(r, V200, c))
        err = 2 + 0.04 * vtrue
        os.makedirs("synthetic", exist_ok=True)
        path = os.path.join("synthetic", f"{name}_rotmod.dat")
        np.savetxt(path, np.c_[r, vtrue + rng.normal(0, err), err, vgas, vdisk, 0 * r], header="Distance = 10 Mpc")
        out[name] = load_rotmod(path)
    return out

HAVE_SPARC = get_sparc()
if HAVE_SPARC:
    GAL = {}
    for f in _files():
        g = load_rotmod(f)
        if len(g["r"]) >= 5:
            GAL[g["name"]] = g
    Q = get_quality()
    if Q:
        GAL = {k: v for k, v in GAL.items() if k in Q and Q[k]["Q"] < 3 and Q[k]["Inc"] >= 30}
        print("Corte de calidad aplicado (Q<3, inc≥30°).")
    else:
        print("Sin tabla de calidad: se usan todas las galaxias.")
else:
    print("⚠️  SIN DATOS REALES: galaxias SINTÉTICAS con halo NFW. Los resultados no tienen valor físico.")
    GAL = make_synthetic_set()

if QUICK and len(GAL) > 40:
    keep = [k for k in ["NGC3198", "NGC2403", "DDO154"] if k in GAL]
    rest = [k for k in GAL if k not in keep]
    GAL = {k: GAL[k] for k in keep + list(RNG.choice(rest, 40 - len(keep), replace=False))}
N_TOT = sum(len(g["r"]) for g in GAL.values())
print(f"Galaxias: {len(GAL)} | puntos: {N_TOT}")

In [ ]:
def get_meta():
    for b in BASES:
        try:
            txt = _get(b + "SPARC_Lelli2016c.mrt").decode("latin-1"); m = {}
            for line in txt.splitlines():
                t = line.split()
                if len(t) >= 18 and t[17] in ("1", "2", "3"):
                    try: m[t[0]] = dict(D=float(t[2]), eD=float(t[3]), inc=float(t[5]), einc=float(t[6]))
                    except ValueError: pass
            if len(m) > 100: return m
        except Exception:
            pass
    return None
META = get_meta() if HAVE_SPARC else None
def meta(k):
    if META and k in META:
        m = META[k]; return max(m["eD"] / m["D"], 0.02), m["inc"], max(m["einc"], 1.0)
    return 0.10, 60.0, 3.0          # valores por defecto (datos sintéticos o sin tabla)
NAMES = list(GAL); NG = len(NAMES)
print(f"Metadatos: {'catálogo SPARC' if META else 'valores por defecto'} | galaxias con bulbo: {sum(np.any(GAL[k]['vbul'] > 0) for k in NAMES)}")

## 2. Tres modelos de masa

| Modelo | Masa del disco | Masa del bulbo | Distancia e inclinación |
|---|---|---|---|
| **M0** (notebooks anteriores) | Υ libre, prior 0.5 ± 0.1 dex | atada: 1.4 × Υ_disco | fijas |
| **M1** | Υ libre, prior 0.5 ± 0.1 dex | **libre**, prior 0.7 ± 0.1 dex | fijas |
| **M2** | igual que M1 | igual que M1 | **libres** con los errores del catálogo |

Al cambiar la distancia por un factor $f$, los radios y $v^2_{bar}$ escalan con $f$ y $g_N$ no cambia, pero $g_{obs} = v^2/r$ escala con $1/f$. Al cambiar la inclinación, la velocidad observada escala con $\sin i / \sin i'$. Igual que antes, cada nuisance se minimiza galaxia a galaxia con su prior (perfil de χ²).

In [ ]:
def law(gN, s, kind):
    gN = np.clip(gN, 1e-16, None); x = np.sqrt(gN / s)
    if kind == "Bose": return gN / (-np.expm1(-x))
    if kind == "simple": return 0.5 * gN * (1 + np.sqrt(1 + 4 * s / gN))
SC = np.geomspace(4e-11, 2.5e-10, 40)
U_D = np.geomspace(0.15, 1.5, 14); U_B = np.geomspace(0.25, 2.0, 6)
K = np.array([-2., -1., 0., 1., 2.])

def galaxy_chi(k, kind, model, u0=0.5, S=SC, floor=0.05, best=False):
    g = GAL[k]; r = g["r"]; sD, inc, einc = meta(k)
    vg2, vd2, vb2 = g["vgas2"], g["vdisk"] * np.abs(g["vdisk"]), g["vbul"] * np.abs(g["vbul"])
    has_b = np.any(vb2 > 0)
    Ud = U_D[:, None]; pd_ = (np.log10(U_D / u0) / 0.1) ** 2
    if model == "M0" or not has_b:
        Ub = (1.4 * U_D if model == "M0" else 0.7 + 0 * U_D)[:, None]; pb = np.zeros(1); nb = 1; tied = True
    else:
        Ub = U_B[None, :]; pb = (np.log10(U_B / 0.7) / 0.1) ** 2; nb = len(U_B); tied = False
    kD = K if model == "M2" else np.zeros(1); kI = K if model == "M2" else np.zeros(1)
    fD = np.exp(kD * sD); inc2 = np.clip(inc + kI * einc, 15, 90); fs = np.sin(np.radians(inc)) / np.sin(np.radians(inc2))
    # g_N por (Υd, Υb, r)
    if tied: gN = ((vg2[None, :] + Ud * vd2[None, :] + Ub * vb2[None, :]) / r * TO_MS2)[:, None, :]
    else: gN = (vg2[None, None, :] + Ud[:, :, None] * vd2 + Ub[:, :, None] * vb2) / r * TO_MS2
    a = law(gN[:, :, None, :], S[None, None, :, None], kind)                       # (A,B,S,N)
    vmod = np.sqrt(np.clip(a[:, :, None, None] * (r * fD[:, None, None, None] / TO_MS2), 0, None))  # (A,B,C,1,S,N)
    vob = (g["vobs"] * fs[:, None])[None, None, None, :, None, :]
    er = np.sqrt((g["err"] * fs[:, None]) ** 2 + (floor * g["vobs"] * fs[:, None]) ** 2)[None, None, None, :, None, :]
    c = (((vob - vmod) / er) ** 2).sum(-1)                                          # (A,B,C,E,S)
    pri = (pd_[:, None, None, None] + pb[None, :, None, None] + (kD ** 2)[None, None, :, None] + (kI ** 2)[None, None, None, :])
    t = c + pri[..., None]
    flat = t.reshape(-1, len(S)); i = flat.argmin(0)
    tot = flat[i, np.arange(len(S))]; dat = c.reshape(-1, len(S))[i, np.arange(len(S))]
    if not best: return tot, dat
    A, B, C_, E_ = np.unravel_index(i[0], t.shape[:4])
    gNb = gN[A, min(B, gN.shape[1] - 1)]; ab = law(gNb, S[0], kind)
    gobs = (g["vobs"] * fs[E_]) ** 2 / (r * fD[C_]) * TO_MS2
    return gNb, gobs, ab, dict(Ud=U_D[A], Ub=(1.4 * U_D[A] if model == "M0" else (0.7 if tied else U_B[B])), fD=fD[C_], dinc=inc2[E_] - inc)

def table(kind, model, u0=0.5):
    T = np.zeros((NG, len(SC))); D = np.zeros_like(T)
    for n, k in enumerate(NAMES): T[n], D[n] = galaxy_chi(k, kind, model, u0)
    return dict(tot=T, dat=D)

def at(tab, s, which="dat"):
    j = np.clip(np.searchsorted(SC, s), 1, len(SC) - 1); w = (np.log(s) - np.log(SC[j - 1])) / (np.log(SC[j]) - np.log(SC[j - 1]))
    return tab[which][:, j - 1] * (1 - w) + tab[which][:, j] * w

t0 = time.time(); TABS = {}
for model in ["M0", "M1", "M2"]:
    for kind in ["Bose", "simple"]:
        TABS[(kind, model, 0.5)] = table(kind, model); print(f"  {kind:6s} {model}: {time.time()-t0:5.0f} s")
for u0 in [0.4, 0.6]:
    TABS[("Bose", "M2", u0)] = table("Bose", "M2", u0); print(f"  Bose   M2 (Υ={u0}): {time.time()-t0:5.0f} s")
NP = sum(len(GAL[k]["r"]) for k in NAMES)

## 3. ¿Adónde se mueve la escala al refinar el modelo de masa?

In [ ]:
def best_s(tab, idx=None, nboot=400):
    T = tab["tot"] if idx is None else tab["tot"][idx]
    j = int(np.argmin(T.sum(0)))
    bs = [SC[np.argmin(T[RNG.integers(0, len(T), len(T))].sum(0))] for _ in range(nboot)]
    return SC[j], np.percentile(bs, 16), np.percentile(bs, 84)

rows = []
for (kind, model, u0), tab in TABS.items():
    s, lo, hi = best_s(tab); chi = tab["dat"][:, int(np.argmin(tab["tot"].sum(0)))].sum()
    rows.append(dict(ley=kind, modelo=model, Υ_prior=u0, s=f"{s:.2e}", IC68=f"{lo:.2e}–{hi:.2e}", chi2=round(chi),
                     s_entre_aΛ2π=round(s / S_LAMBDA, 2), s_entre_cH0_2π=round(s / S_H0, 2),
                     Δχ2_Λ_vs_H=round((at(tab, S_LAMBDA, "tot") - at(tab, S_H0, "tot")).sum(), 1)))
T3 = pd.DataFrame(rows)
print(T3.to_string(index=False))
print("\nΔχ2_Λ_vs_H < 0: los datos locales prefieren s = a_Λ/2π; > 0: prefieren cH₀/2π.")

### ¿Desaparece la ondulación?

Residuos de la ley de Bose (cada modelo de masa en su escala ajustada) por intervalos de aceleración. Si la ondulación del notebook 12 venía del modelo de masa, debería aplanarse de M0 a M2.

In [ ]:
bins = np.linspace(-12, -8.5, 15); mid = 0.5 * (bins[1:] + bins[:-1])
fig, ax = plt.subplots(figsize=(8.5, 4))
for model, col in [("M0", "#9a9a92"), ("M1", "#2a78d6"), ("M2", "#1baf7a")]:
    s = best_s(TABS[("Bose", model, 0.5)], nboot=1)[0]; xs, rs = [], []
    for k in NAMES:
        gNb, gobs, ab, _ = galaxy_chi(k, "Bose", model, S=np.array([s]), best=True)
        ok = (gobs > 0) & (gNb > 0); xs.append(np.log10(gNb[ok])); rs.append(np.log10(gobs[ok] / ab[ok]))
    xs, rs = np.concatenate(xs), np.concatenate(rs)
    med = [np.median(rs[(xs >= lo) & (xs < hi)]) if np.sum((xs >= lo) & (xs < hi)) > 15 else np.nan for lo, hi in zip(bins[:-1], bins[1:])]
    ax.plot(mid, med, "o-", ms=4, lw=2, color=col, label=f"{model} (s = {s:.2e})")
ax.axhline(0, color="#c3c2b7", lw=1)
ax.set_xlabel("log10 g_N [m/s²]  (derecha = gravedad fuerte)"); ax.set_ylabel("mediana log10 (g_obs / g_Bose)")
ax.set_title("Residuos de la ley de Bose según el modelo de masa", loc="left"); ax.legend(fontsize=9)
plt.show()

## 4. Parte B: la escala a redshift intermedio (MUSE-DARK 2026)

**Datos.** Cuatro intervalos de redshift de MUSE-DARK III (Ciocan et al. 2026, A&A 709, L16; 79 galaxias, 0.33 < z < 1.44), ajustados con la misma forma funcional que nuestra ley de Bose. Los valores extremos están en el texto del artículo; los intermedios están digitalizados de su figura 3 (tabla 1 de Alcock 2026, arXiv:2609.38796). **No se sabe si las barras son 1σ o 95 %**, así que se analizan las dos lecturas.

**La comparación honesta es de forma, no de amplitud.** MUSE-DARK estima las masas estelares con otro método que SPARC (sus Υ salen más bajas), y un desplazamiento común de masa cambia la amplitud de $s$ pero no su forma con z. Por eso cada variante se ajusta con **una amplitud libre** $f$:
- **Λ:** $s(z) = f \cdot a_\Lambda/2\pi$ (constante)
- **H:** $s(z) = f \cdot cH(z)/2\pi$
- **Potencia (referencia, 2 parámetros):** $s(z) = A\,(1+z)^\gamma$

Si las masas estuvieran bien calibradas entre muestras, $f$ debería salir ≈ 1.

**Criterio (fijado antes de ejecutar esta celda, aunque con los datos ya vistos):** la variante Λ queda **desfavorecida** si, con las dos lecturas de las barras, su ajuste tiene p < 0.003 (3σ) y el de la variante H tiene p > 0.05.

Puedes añadir más datos en `EXTRA` (z, s en 10⁻¹⁰ m/s², error).

In [ ]:
from scipy.optimize import minimize_scalar, minimize
from scipy.stats import chi2 as CHI2
MUSE = pd.DataFrame(dict(z=[0.503, 0.825, 1.047, 1.276], s=[1.990, 2.200, 2.571, 2.710],
                         lo=[0.085, 0.105, 0.114, 0.134], hi=[0.086, 0.105, 0.110, 0.138]))
EXTRA = []          # p. ej. [(2.2, 3.4, 0.5)]  ← (z, s[1e-10 m/s²], error 1σ)
DZ = MUSE.assign(e=(MUSE.lo + MUSE.hi) / 2)[["z", "s", "e"]]
if EXTRA: DZ = pd.concat([DZ, pd.DataFrame(EXTRA, columns=["z", "s", "e"])], ignore_index=True)

def fit_shapes(sig_scale):
    z, y, e = DZ.z.values, DZ.s.values * 1e-10, DZ.e.values * 1e-10 * sig_scale
    out = {}
    for nm, shape in [("Λ (constante)", lambda z: S_LAMBDA + 0 * z), ("H (∝ H(z))", lambda z: S_H0 * E(z))]:
        m = shape(z); f = np.sum(y * m / e**2) / np.sum(m**2 / e**2); c = np.sum(((y - f * m) / e) ** 2)
        out[nm] = dict(f=f, chi2=c, dof=len(z) - 1, p=CHI2.sf(c, len(z) - 1))
    res = minimize(lambda p: np.sum(((y - p[0] * 1e-10 * (1 + z) ** p[1]) / e) ** 2), [1.5, 0.8])
    out["potencia (1+z)^γ"] = dict(f=res.x[0], gamma=res.x[1], chi2=res.fun, dof=len(z) - 2, p=CHI2.sf(res.fun, max(len(z) - 2, 1)))
    return out

rows = []
for lab, sc in [("barras = 1σ", 1.0), ("barras = 95 %", 1 / 1.96)]:
    for nm, d in fit_shapes(sc).items():
        rows.append(dict(lectura=lab, variante=nm, amplitud_f=(f"{d['f']:.2f}" if "γ" not in nm else f"A={d['f']:.2f}e-10, γ={d['gamma']:.2f}"),
                         chi2=round(d["chi2"], 1), dof=d["dof"], p=f"{d['p']:.1e}"))
T4 = pd.DataFrame(rows); print(T4.to_string(index=False))

### Qué sesgo de masa haría falta para salvar la variante Λ

En la zona de gravedad débil (la mayoría de los puntos de MUSE-DARK), $g_{obs} \approx \sqrt{s\,g_N}$, así que a $g_{obs}$ fijo la escala inferida va como $1/M_{bar}$. Para que la escala medida fuera constante, la masa bariónica tendría que estar **infraestimada** en las cantidades de la tabla. Las incertidumbres típicas de masas estelares a z ~ 1 son de 0.2–0.3 dex, pero aquí lo que importa es el sesgo **que cambia con z** dentro de una misma muestra.

In [ ]:
ref = DZ.s.values[0]
rows = [dict(z=z, s_medida=f"{s:.2f}e-10", sesgo_vs_primer_intervalo=f"{np.log10(s/ref):+.2f} dex",
             sesgo_vs_aΛ_2π=f"{np.log10(s*1e-10/S_LAMBDA):+.2f} dex", sesgo_vs_cH_z_2π=f"{np.log10(s*1e-10/(S_H0*E(z))):+.2f} dex")
        for z, s in zip(DZ.z, DZ.s)]
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
s_loc = {u0: best_s(TABS[("Bose", "M2", u0)], nboot=200) for u0 in [0.4, 0.5, 0.6]}
zz = np.linspace(0, 3, 200); fits = fit_shapes(1.0)
fig, ax = plt.subplots(figsize=(8.5, 4.6))
ax.plot(zz, S_LAMBDA + 0 * zz, color="#eb6834", lw=2, label="Λ: a_Λ/2π (sin ajustar)")
ax.plot(zz, S_H0 * E(zz), color="#2a78d6", lw=2, label="H: cH(z)/2π (sin ajustar)")
ax.plot(zz, fits["H (∝ H(z))"]["f"] * S_H0 * E(zz), color="#2a78d6", lw=1.5, ls="--", label=f"H con amplitud libre (f = {fits['H (∝ H(z))']['f']:.2f})")
ax.plot(zz, fits["Λ (constante)"]["f"] * S_LAMBDA + 0 * zz, color="#eb6834", lw=1.5, ls="--", label=f"Λ con amplitud libre (f = {fits['Λ (constante)']['f']:.2f})")
ax.errorbar(DZ.z, DZ.s * 1e-10, yerr=DZ.e * 1e-10, fmt="o", color="#1f1f1c", ms=6, capsize=3, label="MUSE-DARK 2026 (4 intervalos)")
lo, hi = min(v[1] for v in s_loc.values()), max(v[2] for v in s_loc.values())
ax.errorbar([0], [s_loc[0.5][0]], yerr=[[s_loc[0.5][0] - lo], [hi - s_loc[0.5][0]]], fmt="s", color="#1baf7a", ms=7, capsize=3,
            label="SPARC, este trabajo (M2; barra = rango con Υ 0.4–0.6)")
ax.set_xlabel("redshift z"); ax.set_ylabel("escala s [m/s²]"); ax.set_ylim(0, 4.5e-10)
ax.set_title("La escala de la ley térmica a lo largo del tiempo cósmico", loc="left"); ax.legend(fontsize=8, loc="upper left")
plt.show()

## 5. Veredicto de la parte B

In [ ]:
def veredicto_z():
    ok = []
    for lab, sc in [("1σ", 1.0), ("95 %", 1 / 1.96)]:
        f = fit_shapes(sc); pL, pH = f["Λ (constante)"]["p"], f["H (∝ H(z))"]["p"]
        print(f"Barras = {lab}: p(Λ constante) = {pL:.1e} | p(H) = {pH:.1e} | amplitud H f = {f['H (∝ H(z))']['f']:.2f}")
        ok.append((pL < 0.003, pH > 0.05))
    print()
    if all(a for a, _ in ok) and all(b for _, b in ok):
        print("→ LA VARIANTE Λ (ESCALA CONSTANTE) QUEDA DESFAVORECIDA Y LA VARIANTE H ES COMPATIBLE: la forma de s(z) sigue a H(z). "
              "El horizonte relevante sería el horizonte aparente del universo en expansión, no el de de Sitter.")
    elif all(a for a, _ in ok):
        print("→ LA ESCALA CONSTANTE QUEDA DESFAVORECIDA, PERO H(z) TAMPOCO ENCAJA BIEN con alguna lectura de las barras: "
              "la escala crece con z con otra forma, o hay sistemáticos que dependen de z.")
    elif not any(a for a, _ in ok):
        print("→ LOS DATOS NO EXCLUYEN UNA ESCALA CONSTANTE con la precisión actual.")
    else:
        print("→ RESULTADO DEPENDIENTE DE LA LECTURA DE LAS BARRAS: hace falta la tabla original con errores definidos.")
    print("\nRecuerda: 4 puntos digitalizados, con calibración común (no independientes) y sin sistemáticos que dependan de z. "
          "La amplitud f ≠ 1 refleja que MUSE-DARK y SPARC calibran las masas de forma distinta.")
veredicto_z()

## 6. Parte C: predicciones para datos futuros (esta sí es a ciegas)

Para cada variante, sin ajustar nada: la escala, y su efecto en la relación de Tully–Fisher bariónica, $v_f^4 = G M_{bar}\, s$. A masa fija, $\Delta\log v_f = \tfrac14\Delta\log s$; a velocidad fija, $\Delta\log M_{bar} = -\Delta\log s$.

**No añadimos más variantes después de ver los datos:** solo Λ y H, las dos que salen de un horizonte.

In [ ]:
zs = np.array([0, 0.5, 1, 1.5, 2, 3])
P = pd.DataFrame(dict(z=zs, E_z=np.round(E(zs), 2),
                      s_Λ=[f"{S_LAMBDA:.2e}"] * len(zs), s_H=[f"{v:.2e}" for v in S_H0 * E(zs)],
                      Δlog_v_fija_M_H=np.round(0.25 * np.log10(E(zs)), 3), Δlog_M_fija_v_H=np.round(-np.log10(E(zs)), 2)))
print(P.to_string(index=False))
print("\nVariante Λ: todas las Δ = 0 (relación de Tully–Fisher sin evolución por la escala).")

## 7. Lectura para la hipótesis

- **Parte A** dice cuánto depende la escala local de cómo modelamos la masa. Si con M2 la escala se acerca a $cH_0/2\pi$, la preferencia local por $a_\Lambda/2\pi$ del notebook 12 era un efecto del modelo de masa.
- **Parte B**, si la escala crece con z como H(z): el horizonte que "fija" las galaxias no sería el de la energía oscura pura, sino el **horizonte aparente** del universo en cada época. Su temperatura, $\hbar H/2\pi$, es la que usa la termodinámica de las ecuaciones de Friedmann (Cai y Kim 2005). La idea de fondo, entrelazamiento → temperatura de horizonte → ley térmica de Bose, sobrevive; cambia qué horizonte.
- **Lo que falta:** (1) derivar el factor de Bose; (2) explicar por qué la amplitud absoluta no cuadra entre muestras (¿calibración de masas, o física?); (3) que un sondeo futuro mida s(z) a z ≈ 2–3 con masas homogéneas, donde las dos variantes difieren en un factor 3–5.